In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
#import pytensor.tensor as pt

np.random.seed(42)

In [ ]:
# goal:  build a multi-variable Bayesian regression (`Flow ~ Rainfall + Temperature`) and use `pm.MutableData` to predict flow for new weather conditions

# Mock ERA5-like data: River flow depends on rainfall + temperature
n = 60
rainfall = np.random.uniform(5, 80, n)      # mm/day
temperature = np.random.uniform(5, 35, n)    # °C

# i am god; flow inc w/ rain and dec with temp (evapotrans)

TRUE_B_RAIN = 1.2       # m³/s per mm rainfall
TRUE_B_TEMP = -0.3      # m³/s per °C
TRUE_INTERCEPT = 10.0
TRUE_SIGMA = 5.0

flow = TRUE_INTERCEPT + TRUE_B_RAIN * rainfall + TRUE_B_TEMP * temperature + \
       np.random.normal(0, TRUE_SIGMA, n)  # what that slash doing? continue to the next line type, cant add anything behind it, not even space

In [ ]:
with pm.Model() as hydro_model:

    # mutable data clarification req, why use, when use,
    # so that we can swap these data to train on next set of rain/temp data, while everything remains same
    # pm.Data is a PyTensor Variable that can be updated in-place (mutable). Use it to separate training operations from inference/prediction operations safely.

    rain_data = pm.Data("rain_data", rainfall)
    temp_data = pm.Data("temp_data", temperature)

    # VITAL: To avoid out-of-sample shape mismatches when using `.set_data()`, the target variable must also be wrapped in pm.Data so the entire tensor graph can be resized correctly later using .set_data(). Dummy sizes are required.
    flow_data = pm.Data("flow_data", flow)

# intro log normal here cuz rain or flow(intercept) cant be negative
    b_rain = pm.Normal("b_rain", mu=1.0, sigma=2.0)
    b_temp = pm.Normal("b_temp", mu=0.0, sigma=2.0)
    intercept = pm.Normal("intercept", mu=10, sigma=20)
    sigma = pm.HalfNormal("sigma", sigma=10)

    expected_flow = pm.Deterministic("expected_flow", intercept + b_rain * rain_data + b_temp * temp_data)

    # as exp flow is neg, pos approaches to 0, if we use mod, a drought gonna magically be a flood

    # to upgrade the model and have strictly positive values, we use gamma and then shrink priors to match the log/exp scale but for now we gauss

   # expected_flow_pos = pt.exp(expected_flow)

   # also alt parameterization, behind the bb, pymc auto converts mu and sigma passing into alpha and beta if gamma used

    y = pm.Normal("y", mu=expected_flow, sigma =sigma, observed=flow_data)

    prior_checks = pm.sample_prior_predictive(samples=2000, random_seed=42)

    trace = pm.sample(2000, tune=1000, cores=1, chains=1, random_seed=42, progressbar=False)
    ppc = pm.sample_posterior_predictive(trace=trace, random_seed=42, progressbar=False)


fig, ax = plt.subplots(figsize=(12, 6))
az.plot_ppc(prior_checks,
group="prior",
kind="kde",
ax=ax,
colors=['gray', 'black', 'blue'], alpha=0.8)
az.plot_kde(flow, ax=ax, plot_kwargs={"color": "red", "linewidth": 3, "linestyle": "--"}, label="Actual Observed Data")
plt.title("Prior Predictive Check: Physicality Audit")
plt.tight_layout()
plt.show()

print(az.summary(trace, var_names=["b_rain", "b_temp", "intercept", "sigma"]))
az.plot_ppc(ppc, observed_rug=True)
plt.title("Posterior Predictive Check: Flow Model")
plt.tight_layout()
plt.show()

display(pm.model_to_graphviz(hydro_model))




In [ ]:
new_rainfall = np.array([100, 120, 150, 80, 60])
new_temperature = np.array([25, 28, 30, 22, 20])


# 1. REPLACE OR APPEND? They REPLACE the 60 observations. pm.set_data() swaps the underlying memory buffer.
# 2. generated a Posterior Distribution—thousands of validated physical scenarios (`trace`) that explain the 60-day historical reality.
# 3. You train on HISTORY (where you know the actual flow) to calibrate the physics. You predict on the FUTURE (new_rainfall) where the flow is unknown. This is fundamentally how risk forecasting works.

with hydro_model:

    # The original observed variable `flow_data` was set to shape (60,). When passing new data of shape (5,),
    # expected_flow becomes size 5, clashing with the observed=flow_data (60).
    # The correct physical engineering approach is to also replace the observed data with a dummy array of the right shape
    # so the PyMC graph compiles without shape conflicts during predictive sampling.
    dummy_flow_target = np.zeros_like(new_rainfall)  # Array of 5 zeros to satisfy PyMC's size constraints for the response

    pm.set_data({
        "rain_data": new_rainfall,
        "temp_data": new_temperature,
        "flow_data": dummy_flow_target
    })

    # Generate predictive flow scenarios for the 5 new weather conditions
    predictions = pm.sample_posterior_predictive(
        trace,
        predictions=True, # Explicitly tell PyMC we are generating out-of-sample predictions i.e new data model has never seen before
        random_seed=42
    )


pred_flow = predictions.predictions["y"].values.reshape(-1, len(new_rainfall))

print(f"Prediction Matrix Shape: {pred_flow.shape} (2000 simulations per 5 days)")

print("Predicted flow for extreme rainfall scenarios:")
for i, (r, t) in enumerate(zip(new_rainfall, new_temperature)):
    mean_pred = pred_flow[:, i].mean()
    hdi = np.percentile(pred_flow[:, i], [3, 97])
    print(f"  Rain={r}mm, Temp={t}°C → Flow={mean_pred:.1f} m³/s (94% HDI: {hdi[0]:.1f}-{hdi[1]:.1f})")


In [ ]:
with pm.Model() as simple_model:
    rain_data = pm.Data("rain_data", rainfall)
    b_rain = pm.Normal("b_rain", mu=1.0, sigma=2.0)
    intercept = pm.Normal("intercept", mu=10, sigma=20)
    sigma = pm.HalfNormal("sigma", sigma=10)
    mu =pm.Deterministic("mu", intercept + b_rain * rain_data)
    y = pm.Normal("y", mu=mu, sigma=sigma, observed=flow)
    trace_simple = pm.sample(2000, tune=1000, cores=1, random_seed=42)

# Compare sigma: smaller sigma = better model
sigma_full = trace.posterior["sigma"].values.flatten().mean()
sigma_simple = trace_simple.posterior["sigma"].values.flatten().mean()
print(f"Full model sigma (Rain+Temp): {sigma_full:.2f}")
print(f"Simple model sigma (Rain only): {sigma_simple:.2f}")
print(f"Adding temperature {'REDUCED' if sigma_full < sigma_simple else 'DID NOT REDUCE'} unexplained noise")

In [ ]:
# Rather than just comparing the sigma (which is an in-sample metric that always favors complex models,
# risking overfitting), az.compare uses Leave-One-Out cross-validation to mathematically project which model will survive reality better.

with hydro_model:
    # Reset data back to the training shape (60) before LOO evaluation
    pm.set_data({
        "rain_data": rainfall,
        "temp_data": temperature,
        "flow_data": flow
    })
    pm.compute_log_likelihood(trace, progressbar=False)

    # log cuz we dealing with small probab, rather than mult very smal numbers, we add negative numbers which is stable
with simple_model:
    pm.compute_log_likelihood(trace_simple, progressbar=False)

# Run LOO comparison (Expected Log Predictive Density). HIGHER ELPD is better.

model_comparison = az.compare({
    "Full Model (Rain+Temp)": trace,
    "Simple Model (Rain Only)": trace_simple
}, ic="loo", scale="deviance") # scale='deviance' means lower deviance is better, or default scale='log' means higher is better.

#display(model_comparison)

fig, ax = plt.subplots(figsize=(10, 4))
az.plot_compare(model_comparison, ax=ax)
plt.title("Leave-One-Out (LOO) Predictive Accuracy Audit")
plt.tight_layout()
plt.show()

# If the 'Simple Model' has a higher expected log predictive density (ELPD), the 'Full Model' is overfitting and mathematically dangerous.


In [ ]:
#: EPISTEMIC vs ALEATORIC RATIO

# 1. Calculate Aleatoric Variance (V_A)
# nature noise -> mean of posterior sigma -> square

mean_sigma = trace.posterior["sigma"].values.mean()
V_A = mean_sigma ** 2

# 2. Calculate Epistemic Variance (V_E) at a specific condition
# Let's say we are forecasting for a day with 50mm rain and 20C temp.
# We calculate the EXPECTED mean flow (the deterministic engine) for all 2000 drawn scenarios.
target_rain = 50.0
target_temp = 20.0

# Extract the flattened parameter distributions arrays
b_rain_samples = trace.posterior["b_rain"].values.flatten()
b_temp_samples = trace.posterior["b_temp"].values.flatten()
intercept_samples = trace.posterior["intercept"].values.flatten()

# Calculate the mean flow for all 2000 versions of the rules
expected_flows = intercept_samples + (b_rain_samples * target_rain) + (b_temp_samples * target_temp)

# The variance of these 2000 predicted means is our Epistemic Variance.
# It represents how much the model disagrees with ITSELF about the physics.
V_E = np.var(expected_flows)

# 3. The Ratio
ratio = V_E / V_A

print(f"Aleatoric Variance (V_A): {V_A:.2f} (The Mountain's Chaos)")
print(f"Epistemic Variance (V_E): {V_E:.2f} (Your Ignorance)")
print(f"Ratio (V_E / V_A): {ratio:.3f}")

if ratio > 1:
    print("Verdict: EPISTEMIC DOMINANT. You lack data. Do not sign the Fixed-Price contract.")
elif ratio < 0.1:
    print("Verdict: ALEATORIC DOMINANT. Fan has collapsed. Stop drilling, start building.")
else:
    print("Verdict: MARGINAL. Proceed with localized physical verification.")
